# Partie 3 - CNN, classification d'images et transfer learning

Notebook de travail pour les phases 1.1 a 3.5 du sujet.

## Phase 1.1 - Setup Colab et organisation du dataset

Objectif : telecharger le dataset, organiser `train/` et `val/` par classe, verifier les comptes et afficher une grille 2x3.

In [ ]:
CLASS_A = "cat"
CLASS_B = "dog"
DATA_ROOT = "data"
IMG_SIZE = (128, 128)
IMG_SIZE_TL = (160, 160)
BATCH_SIZE = 32
SEED = 42

## Phase 1.2 - Preprocessing, normalisation et batching

In [ ]:
import os
import tensorflow as tf

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "train"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=SEED,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "val"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False,
)

normalization_layer = tf.keras.layers.Rescaling(1.0 / 255)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda images, labels: (normalization_layer(images), labels)).cache().prefetch(AUTOTUNE)
val_ds = val_ds.map(lambda images, labels: (normalization_layer(images), labels)).cache().prefetch(AUTOTUNE)

## Phase 1.3 - Architecture CNN from scratch

In [ ]:
from tensorflow.keras import layers, models

def build_cnn_scratch(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    return model

model_scratch = build_cnn_scratch(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
model_scratch.summary()

## Phase 1.4 - Entrainement scratch et diagnostic overfitting

In [ ]:
import datetime
import time
import matplotlib.pyplot as plt

model_scratch.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
log_dir = "logs/scratch/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

start = time.time()
history_scratch = model_scratch.fit(train_ds, epochs=20, validation_data=val_ds, callbacks=[tensorboard_callback, early_stopping])
training_time_scratch = time.time() - start

## Phase 2 - Augmentation, Dropout et comparaison TP1/TP2

In [ ]:
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

def build_cnn_augmented(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        data_augmentation,
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dropout(0.4),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    return model

## Phase 3 - MobileNetV2, fine-tuning, comparaison et export TFLite

In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
train_ds_tl = tf.keras.utils.image_dataset_from_directory(os.path.join(DATA_ROOT, "train"), image_size=IMG_SIZE_TL, batch_size=BATCH_SIZE, label_mode="binary", shuffle=True, seed=SEED)
val_ds_tl = tf.keras.utils.image_dataset_from_directory(os.path.join(DATA_ROOT, "val"), image_size=IMG_SIZE_TL, batch_size=BATCH_SIZE, label_mode="binary", shuffle=False)
train_ds_tl = train_ds_tl.map(lambda images, labels: (preprocess_input(images), labels)).prefetch(AUTOTUNE)
val_ds_tl = val_ds_tl.map(lambda images, labels: (preprocess_input(images), labels)).prefetch(AUTOTUNE)

base_model = tf.keras.applications.MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights="imagenet")
base_model.trainable = False
inputs = tf.keras.Input(shape=(160, 160, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
model_tl = tf.keras.Model(inputs, outputs)
model_tl.summary()